# ============================================================
# HRV Analysis & MS Monitoring Notebook
# Author: <Your Name>
# Purpose:
#   - Analyze HRV data using Python
#   - Interpret results through a Multiple Sclerosis (MS) lens
#   - Produce clinically cautious, reproducible insights
#
# NOTE:
#   This notebook is for monitoring & research purposes only.
#   It is NOT a diagnostic or clinical decision tool.
# ============================================================


# ============================================================
# 1. IMPORTS & ENVIRONMENT SETUP
# ============================================================

import pandas as pd
import numpy as np

from scipy import stats
from scipy.signal import welch

import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime, timedelta

# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Visualization defaults
plt.rcParams["figure.figsize"] = (12, 6)


# ============================================================
# 2. DOMAIN CONTEXT & ASSUMPTIONS (READ CAREFULLY)
# ============================================================

"""
HRV CONTEXT
-----------
- HRV reflects autonomic nervous system (ANS) balance
- RMSSD → Parasympathetic activity (most reliable for wearables)
- SDNN  → Overall variability (context-dependent)
- LF/HF → Controversial, interpret cautiously

MS CONTEXT
----------
- MS patients may show:
  - Reduced baseline HRV
  - Blunted recovery after stress
  - Higher sensitivity to fatigue, heat, infection
- HRV changes may reflect:
  - Autonomic dysfunction
  - Fatigue load
  - Pseudo-relapse conditions
  - Illness or medication effects

CLINICAL SAFETY
---------------
- HRV trends > single values
- Always contextualize with:
  - Sleep quality
  - Illness/infection
  - Heat exposure
  - Physical & cognitive load
"""


# ============================================================
# 3. DATA INGESTION
# ============================================================

"""
Expected data sources:
- RR intervals (preferred) OR
- Device-calculated HRV metrics (fallback)

Example inputs:
- Garmin CSV exports
- API-derived daily summaries
"""

# Example: Load RR interval data
# Replace with your actual file path
rr_file_path = "data/rr_intervals.csv"

# Expected columns:
# timestamp | rr_ms
try:
    rr_df = pd.read_csv(rr_file_path, parse_dates=["timestamp"])
except FileNotFoundError:
    rr_df = None
    print("RR interval file not found. Proceeding without RR-level data.")


# Example: Load daily HRV summary
daily_hrv_path = "data/daily_hrv.csv"

# Expected columns:
# date | rmssd | sdnn | resting_hr | sleep_score | body_battery
try:
    daily_df = pd.read_csv(daily_hrv_path, parse_dates=["date"])
except FileNotFoundError:
    daily_df = None
    print("Daily HRV file not found.")


# ============================================================
# 4. DATA QUALITY & PREPROCESSING
# ============================================================

def clean_rr_intervals(rr_df, min_rr=300, max_rr=2000):
    """
    Clean RR intervals:
    - Remove physiologically implausible values
    - Remove NaNs
    """
    rr = rr_df.copy()
    rr = rr.dropna(subset=["rr_ms"])
    rr = rr[(rr["rr_ms"] >= min_rr) & (rr["rr_ms"] <= max_rr)]
    return rr


if rr_df is not None:
    rr_df = clean_rr_intervals(rr_df)


# ============================================================
# 5. HRV METRIC CALCULATION
# ============================================================

def calculate_time_domain_hrv(rr_ms):
    """
    Calculate standard time-domain HRV metrics.
    """
    diff_rr = np.diff(rr_ms)

    metrics = {
        "mean_rr": np.mean(rr_ms),
        "sdnn": np.std(rr_ms, ddof=1),
        "rmssd": np.sqrt(np.mean(diff_rr**2)),
        "pnn50": np.sum(np.abs(diff_rr) > 50) / len(diff_rr) * 100
    }

    return metrics


def calculate_frequency_domain_hrv(rr_ms, fs=4.0):
    """
    Frequency domain HRV using Welch method.
    NOTE: Interpret LF/HF with caution in MS.
    """
    rr_sec = rr_ms / 1000.0
    rr_interp = np.interp(
        np.arange(0, len(rr_sec)),
        np.arange(0, len(rr_sec)),
        rr_sec
    )

    f, pxx = welch(rr_interp, fs=fs)

    lf_band = (0.04, 0.15)
    hf_band = (0.15, 0.40)

    lf_power = np.trapz(pxx[(f >= lf_band[0]) & (f <= lf_band[1])])
    hf_power = np.trapz(pxx[(f >= hf_band[0]) & (f <= hf_band[1])])

    return {
        "lf_power": lf_power,
        "hf_power": hf_power,
        "lf_hf_ratio": lf_power / hf_power if hf_power > 0 else np.nan
    }


if rr_df is not None:
    hrv_time = calculate_time_domain_hrv(rr_df["rr_ms"].values)
    hrv_freq = calculate_frequency_domain_hrv(rr_df["rr_ms"].values)
else:
    hrv_time, hrv_freq = None, None


# ============================================================
# 6. TREND & BASELINE ANALYSIS (MOST IMPORTANT FOR MS)
# ============================================================

def rolling_baseline(series, window=7):
    """
    Rolling baseline to detect meaningful deviations.
    """
    return series.rolling(window=window, min_periods=3).mean()


if daily_df is not None:
    daily_df = daily_df.sort_values("date")
    daily_df["rmssd_baseline"] = rolling_baseline(daily_df["rmssd"])
    daily_df["rmssd_delta"] = daily_df["rmssd"] - daily_df["rmssd_baseline"]


# ============================================================
# 7. VISUALIZATION
# ============================================================

if daily_df is not None:
    plt.plot(daily_df["date"], daily_df["rmssd"], label="RMSSD")
    plt.plot(daily_df["date"], daily_df["rmssd_baseline"], label="7-day Baseline")
    plt.axhline(daily_df["rmssd"].mean(), linestyle="--", alpha=0.5)
    plt.title("HRV (RMSSD) Trend – MS Monitoring Context")
    plt.xlabel("Date")
    plt.ylabel("RMSSD (ms)")
    plt.legend()
    plt.show()


# ============================================================
# 8. MS-SPECIFIC INTERPRETATION LOGIC
# ============================================================

def interpret_rmssd_change(delta):
    """
    MS-aware interpretation of RMSSD changes.
    """
    if delta < -10:
        return "Significant parasympathetic suppression – possible fatigue, illness, heat, or pseudo-relapse"
    elif delta < -5:
        return "Moderate reduction – monitor closely"
    elif delta > 5:
        return "Improved recovery / parasympathetic activation"
    else:
        return "Within normal variability"


if daily_df is not None:
    daily_df["ms_interpretation"] = daily_df["rmssd_delta"].apply(interpret_rmssd_change)


# ============================================================
# 9. ALTERNATIVE ANALYSES (OPTIONAL EXTENSIONS)
# ============================================================

"""
Optional extensions:
- Compare HRV vs sleep score
- HRV vs subjective fatigue
- Change-point detection
- Heat exposure correlation
- Relapse / pseudo-relapse annotation
"""


# ============================================================
# 10. PRACTICAL SUMMARY (AUTO-GENERATED)
# ============================================================

def generate_summary(df):
    latest = df.iloc[-1]
    summary = f"""
    HRV SUMMARY (Latest Day)
    ------------------------
    RMSSD: {latest['rmssd']:.1f} ms
    Baseline: {latest['rmssd_baseline']:.1f} ms
    Delta: {latest['rmssd_delta']:.1f} ms

    Interpretation:
    {latest['ms_interpretation']}

    Recommended Actions:
    - Check sleep quality
    - Review fatigue & heat exposure
    - Avoid overinterpreting single-day drops
    """
    return summary


if daily_df is not None:
    print(generate_summary(daily_df))

# ============================================================
# 11. RELAPSE / EVENT ANNOTATION
# ============================================================

"""
Annotation philosophy:
- User-provided or clinician-confirmed events
- NEVER inferred automatically
- Used only for interpretation and visualization
"""

# Example annotation table
annotations = pd.DataFrame({
    "date": [
        "2025-01-10",
        "2025-02-03"
    ],
    "event_type": [
        "Pseudo-relapse (heat)",
        "Confirmed relapse"
    ],
    "notes": [
        "Heatwave + fatigue",
        "Neurologist-confirmed relapse"
    ]
})

annotations["date"] = pd.to_datetime(annotations["date"])

if daily_df is not None:
    daily_df = daily_df.merge(
        annotations,
        on="date",
        how="left"
    )


# ============================================================
# 12. STATISTICAL ALERTING LOGIC
# ============================================================

def compute_z_score(series, window=30):
    rolling_mean = series.rolling(window, min_periods=10).mean()
    rolling_std = series.rolling(window, min_periods=10).std()
    return (series - rolling_mean) / rolling_std


if daily_df is not None:
    daily_df["rmssd_z"] = compute_z_score(daily_df["rmssd"])

    def alert_logic(z):
        if z <= -2.0:
            return "CRITICAL DROP"
        elif z <= -1.5:
            return "WARNING DROP"
        elif z >= 1.5:
            return "POSITIVE RECOVERY"
        else:
            return "NORMAL"

    daily_df["alert_level"] = daily_df["rmssd_z"].apply(alert_logic)


# ============================================================
# 13. AUTOMATED WEEKLY REPORT
# ============================================================

def generate_weekly_report(df):
    last_7 = df.tail(7)

    report = {
        "avg_rmssd": last_7["rmssd"].mean(),
        "rmssd_trend": last_7["rmssd"].iloc[-1] - last_7["rmssd"].iloc[0],
        "alerts": last_7["alert_level"].value_counts().to_dict(),
        "events": last_7["event_type"].dropna().tolist()
    }

    narrative = f"""
    WEEKLY HRV REPORT
    =================
    Average RMSSD: {report['avg_rmssd']:.1f} ms
    Weekly Trend: {report['rmssd_trend']:.1f} ms

    Alerts:
    {report['alerts']}

    Annotated Events:
    {report['events'] if report['events'] else 'None'}

    Interpretation:
    - Focus on trend, not daily noise
    - Review sleep, fatigue, and heat exposure
    - Escalate only if sustained drops persist >7–10 days
    """

    return narrative


if daily_df is not None:
    print(generate_weekly_report(daily_df))


# ============================================================
# 14. PLOTLY DASHBOARD
# ============================================================

import plotly.express as px

if daily_df is not None:
    fig = px.line(
        daily_df,
        x="date",
        y="rmssd",
        color="alert_level",
        title="HRV RMSSD Trend with Alerts (MS Context)",
        markers=True
    )

    fig.add_scatter(
        x=daily_df["date"],
        y=daily_df["rmssd_baseline"],
        mode="lines",
        name="Baseline",
        line=dict(dash="dash")
    )

    fig.show()


# ============================================================
# 15. EXPORT FOR POWER BI
# ============================================================

if daily_df is not None:
    daily_df.to_csv("output/hrv_ms_monitoring_dataset.csv", index=False)




# ============================================================
# END OF NOTEBOOK
# ============================================================
